# Module 06 — Training Observability

This notebook builds a reusable monitor for loss, gradient norms, weight norms, actual updates, learning rate, step time, throughput, and CUDA peak memory.

## 1. Monitoring utilities

The monitor is loss-agnostic: pass the model, optimizer, batch, and a loss function.

In [ ]:
import time
import torch
import torch.nn as nn


@torch.no_grad()
def global_l2_norm(tensors):
    norms = [
        tensor.detach().float().norm(2)
        for tensor in tensors
        if tensor is not None
    ]
    if not norms:
        return 0.0
    return torch.linalg.vector_norm(torch.stack(norms), ord=2).item()


def cuda_memory_report(device=None):
    if not torch.cuda.is_available():
        return {"device": "CPU", "cuda": False}
    if device is None:
        device = torch.device("cuda")
    gigabyte = 1024 ** 3
    return {
        "device": str(device),
        "cuda": True,
        "allocated_gb": torch.cuda.memory_allocated(device) / gigabyte,
        "reserved_gb": torch.cuda.memory_reserved(device) / gigabyte,
        "peak_allocated_gb": torch.cuda.max_memory_allocated(device) / gigabyte,
        "peak_reserved_gb": torch.cuda.max_memory_reserved(device) / gigabyte,
    }


def monitored_step(
    model,
    optimizer,
    x,
    y,
    loss_fn,
    *,
    max_grad_norm=None,
    measure_updates=True,
):
    model.train()
    device = x.device
    use_cuda = device.type == "cuda"

    if use_cuda:
        torch.cuda.synchronize(device)
        torch.cuda.reset_peak_memory_stats(device)

    start = time.perf_counter()
    optimizer.zero_grad(set_to_none=True)
    parameters = [p for p in model.parameters() if p.requires_grad]
    before = (
        [p.detach().clone() for p in parameters]
        if measure_updates else None
    )

    prediction = model(x)
    loss = loss_fn(prediction, y)
    if not torch.isfinite(loss).item():
        raise FloatingPointError("Non-finite loss")

    loss.backward()
    gradients = [p.grad for p in parameters if p.grad is not None]
    if any(not torch.isfinite(g).all() for g in gradients):
        raise FloatingPointError("Non-finite gradient")

    grad_norm = global_l2_norm(gradients)
    if max_grad_norm is not None:
        torch.nn.utils.clip_grad_norm_(
            parameters,
            max_norm=max_grad_norm,
            error_if_nonfinite=True,
        )

    optimizer.step()
    if any(not torch.isfinite(p).all() for p in parameters):
        raise FloatingPointError("Non-finite parameter")

    if measure_updates:
        weight_norm = global_l2_norm(before)
        update_norm = global_l2_norm(
            p.detach() - old for p, old in zip(parameters, before)
        )
        update_ratio = update_norm / max(weight_norm, 1e-12)
    else:
        weight_norm = float("nan")
        update_norm = float("nan")
        update_ratio = float("nan")

    if use_cuda:
        torch.cuda.synchronize(device)
    elapsed = time.perf_counter() - start

    stats = {
        "loss": loss.item(),
        "grad_norm": grad_norm,
        "weight_norm": weight_norm,
        "update_norm": update_norm,
        "update_ratio": update_ratio,
        "lr": optimizer.param_groups[0]["lr"],
        "step_time_s": elapsed,
        "samples_per_sec": x.size(0) / elapsed,
    }
    if use_cuda:
        stats["peak_allocated_gb"] = (
            torch.cuda.max_memory_allocated(device) / 1024**3
        )
    return stats


print(cuda_memory_report())

## 2. Train a monitored regression model

The model learns `y = 2*x1 - 3*x2`.

In [ ]:
import matplotlib.pyplot as plt
import torch.nn.functional as F

torch.manual_seed(42)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")


class TinyRegressor(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(2, 32),
            nn.ReLU(),
            nn.Linear(32, 1),
        )

    def forward(self, x):
        return self.net(x)


model = TinyRegressor().to(device)
optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=1e-3,
    weight_decay=0.0,
)


def make_batch(batch_size=128):
    x = torch.randn(batch_size, 2, device=device)
    y = 2 * x[:, 0:1] - 3 * x[:, 1:2]
    return x, y


history = []
for step in range(200):
    x, y = make_batch()
    stats = monitored_step(
        model,
        optimizer,
        x,
        y,
        loss_fn=F.mse_loss,
        max_grad_norm=1.0,
    )
    stats["step"] = step
    history.append(stats)

    if step % 20 == 0:
        print(
            f"Step {step:03d} | Loss {stats['loss']:.4f} | "
            f"Grad {stats['grad_norm']:.3f} | "
            f"Update ratio {stats['update_ratio'] * 100:.4f}%"
        )

## 3. Plot the monitoring dashboard

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(12, 8))
steps = [record["step"] for record in history]
metrics = [
    ("loss", "Training loss"),
    ("grad_norm", "Gradient norm before clipping"),
    ("update_ratio", "Update-to-weight ratio"),
    ("weight_norm", "Weight norm"),
]

for axis, (key, label) in zip(axes.flat, metrics):
    axis.plot(steps, [record[key] for record in history])
    axis.set_title(label)
    axis.set_xlabel("Optimizer step")
    axis.grid(alpha=0.25)

plt.tight_layout()
plt.show()

## 4. Controlled learning-rate comparison

Use identical initialization and batches; change only the learning rate.

In [ ]:
import copy

initial_model = TinyRegressor().to(device)
initial_state = copy.deepcopy(initial_model.state_dict())
batches = [make_batch() for _ in range(100)]


def run_experiment(learning_rate):
    experiment_model = TinyRegressor().to(device)
    experiment_model.load_state_dict(initial_state)
    experiment_optimizer = torch.optim.AdamW(
        experiment_model.parameters(),
        lr=learning_rate,
        weight_decay=0.0,
    )
    records = []

    for step, (x, y) in enumerate(batches):
        try:
            stats = monitored_step(
                experiment_model,
                experiment_optimizer,
                x,
                y,
                loss_fn=F.mse_loss,
                max_grad_norm=None,
            )
        except FloatingPointError as error:
            print(f"Stopped at step {step}: {error}")
            break
        records.append({"step": step, **stats})

    return records


baseline = run_experiment(1e-3)
stress = run_experiment(1e-1)

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for records, label in ((baseline, "lr=0.001"), (stress, "lr=0.1")):
    steps = [record["step"] for record in records]
    axes[0].plot(steps, [record["loss"] for record in records], label=label)
    axes[1].plot(steps, [record["grad_norm"] for record in records], label=label)
    axes[2].plot(steps, [record["update_ratio"] for record in records], label=label)

for axis, title in zip(axes, ("Loss", "Gradient norm", "Update / weight")):
    axis.set_title(title)
    axis.set_xlabel("Step")
    axis.legend()
    axis.grid(alpha=0.25)

plt.tight_layout()
plt.show()

## 5. Optional logging and CUDA timing

TensorBoard and CUDA events are optional tools. The cells below only run when their dependencies or hardware are available.

In [ ]:
import math

try:
    from torch.utils.tensorboard import SummaryWriter
except ImportError:
    SummaryWriter = None

if SummaryWriter is not None:
    writer = SummaryWriter("runs/module06")
    for record in history:
        step = record["step"]
        for key in (
            "loss", "grad_norm", "weight_norm",
            "update_norm", "update_ratio", "lr",
            "step_time_s",
        ):
            value = record[key]
            if math.isfinite(value):
                writer.add_scalar(f"train/{key}", value, step)
    writer.flush()
    writer.close()
    print("TensorBoard records written.")
else:
    print("TensorBoard is not installed; skipping logging.")

if torch.cuda.is_available():
    start_event = torch.cuda.Event(enable_timing=True)
    end_event = torch.cuda.Event(enable_timing=True)
    start_event.record()
    output = model(x)
    end_event.record()
    torch.cuda.synchronize()
    print("GPU elapsed milliseconds:", start_event.elapsed_time(end_event))
else:
    print("CUDA is unavailable; skipping CUDA-event timing.")

## Exercises

1. Log validation loss on a fixed diagnostic batch.
2. Change the stress-test learning rate and compare update ratios.
3. Measure peak CUDA memory before and after increasing batch size.
4. Add layer-wise gradient norms every 50 steps instead of every step.